[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C31_Coding_Agent_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯标准库、CPU 可跑、无需 API key**。我们用 **MockLLM**（确定性替身）驱动真实的「LLM ↔ 工具」循环，文件 / shell / pytest 全部在 **`tempfile` 临时目录**里**真实执行**。

这个 notebook 做四件事：① 确认环境（只要标准库）；② 建一个**临时工作区**并在里面真跑 pytest，立下「真实但隔离」的物理基础；③ 写出最小 **MockLLM** 与最小 **agent 循环**，端到端跑通一次「LLM 决定调用工具 → 执行 → 回填」；④ 写出 **无 key 回退**的 `get_llm()` 工厂——有 key 用真 Claude，无 key 用 MockLLM。

## 1 · 环境自检

只需要 Python 标准库。`anthropic` 包是**可选**的——装了且有 `ANTHROPIC_API_KEY` 才会走真实 Claude，否则全程用 MockLLM。

In [ ]:
import sys, platform, os, tempfile, shutil, subprocess, re, difflib
print('Python', sys.version.split()[0], '|', platform.system())
print('标准库就绪：os / subprocess / tempfile / re / difflib ✅')
# 可选：真实 Claude 适配（无则自动回退 MockLLM，不影响课程）
have_key = bool(os.environ.get('ANTHROPIC_API_KEY'))
try:
    import anthropic; have_anthropic = True
except Exception:
    have_anthropic = False
print(f'ANTHROPIC_API_KEY 存在: {have_key} | anthropic 包: {have_anthropic}')
print('→ 本课默认走 MockLLM（确定性、零成本、可复现）；上面两者都为 True 时可切真实 Claude。')
assert sys.version_info >= (3, 8), '建议 Python 3.8+'
print('环境就绪 ✅')

## 2 · 物理基础：临时工作区里【真的】跑 pytest

本课所有副作用操作都关进一个 `tempfile.mkdtemp()` 沙箱：里面的文件、命令、测试都**真实执行**，但与你的系统隔离、用完即焚。

下面建一个**玩具仓库**（一个带 bug 的 `calc.py` + 一个测试），在工作区里真跑 pytest，看着它因 bug 而**失败**——这正是后面 agent 要修的。

In [ ]:
def make_toy_repo():
    '''建一个隔离工作区，放一个【故意有 bug】的 calc.py 和它的测试。返回工作区路径。'''
    work = tempfile.mkdtemp(prefix='agent_ws_')
    with open(os.path.join(work, 'calc.py'), 'w') as f:
        f.write('def add(a, b):\n    return a - b   # BUG: 应当是 a + b\n')
    with open(os.path.join(work, 'test_calc.py'), 'w') as f:
        f.write('from calc import add\n\n'
                'def test_add():\n    assert add(2, 3) == 5\n')
    return work

def run_pytest(work, timeout=60):
    '''在工作区里【真的】跑 pytest，返回 (returncode, 合并输出)。'''
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q'],
                       cwd=work, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr)

work = make_toy_repo()
rc, out = run_pytest(work)
print('pytest 返回码 =', rc, '（非 0 = 有测试失败）')
print('输出片段:', out.strip().splitlines()[-1] if out.strip() else '(空)')
assert rc != 0, 'bug 应当让测试失败'
print('✅ 工作区里真的跑了 pytest，bug 如期让它变红 —— 这就是 agent 的起点')

## 3 · 最小 MockLLM：确定性的「大脑」替身

真实 LLM 又贵又慢又不确定。教学/测试时我们用 **MockLLM**：它不思考，只**按预设脚本**依次返回动作（`('tool', 名字, 参数)` 表示要调用工具；`('final', 文本)` 表示最终答复）。

这样整个 agent 循环就能**零成本、可复现**地端到端跑通——固定模型决策，专注验证*脚手架*是否正确。

In [ ]:
class MockLLM:
    '''确定性 LLM 替身：按 script 依次吐出动作。
       script = [('tool', name, args_dict), ..., ('final', text)]'''
    def __init__(self, script):
        self.script = list(script)
        self.i = 0
    def step(self, messages=None):
        '''返回下一个动作；忽略 messages（真实 LLM 会读它，Mock 不需要）。'''
        if self.i >= len(self.script):
            return ('final', '(脚本结束)')
        action = self.script[self.i]
        self.i += 1
        return action

# 演示：一个会先调用工具、再给最终答复的脚本
llm = MockLLM([('tool', 'read_file', {'path': 'calc.py'}),
               ('final', '我看完了 calc.py')])
print(llm.step())   # ('tool', 'read_file', {...})
print(llm.step())   # ('final', ...)
assert MockLLM([('final','x')]).step() == ('final', 'x')
print('✅ MockLLM 可用：确定性地按脚本驱动 agent，零成本可复现')

## 4 · 最小 agent 循环：把「大脑」和「工具」缝起来

agent 循环 = 反复地：问 LLM 想干什么 → 若要调工具就执行并回填观察 → 直到 LLM 给最终答复或到步数上限。

下面用上面的 MockLLM + 一个**真的会读取工作区文件**的 `read_file` 工具，端到端跑一次。

In [ ]:
def read_file_tool(work, path):
    '''一个真实工具：读取工作区里的文件内容（带行号，对 LLM 友好）。'''
    full = os.path.join(work, path)
    with open(full) as f:
        lines = f.read().splitlines()
    return '\n'.join(f'{i+1:4d} | {ln}' for i, ln in enumerate(lines))

def agent_loop(llm, work, tools, max_steps=10):
    '''最小 agent 主循环。tools: {名字: 函数(work, **args)}。返回 (最终答复, 步数, 观察日志)。'''
    messages = []            # 对话历史（真实 LLM 会读，这里也累积以便接真模型）
    observations = []
    for step in range(max_steps):
        action = llm.step(messages)
        if action[0] == 'final':
            return action[1], step, observations
        _, name, args = action
        assert name in tools, f'未知工具 {name}'
        result = tools[name](work, **args)         # 【真的】执行工具
        observations.append((name, args, result))
        messages.append({'role': 'assistant', 'content': f'call {name}({args})'})
        messages.append({'role': 'user', 'content': f'[tool_result]\n{result}'})
    return '(到达步数上限)', max_steps, observations

tools = {'read_file': read_file_tool}
llm = MockLLM([('tool', 'read_file', {'path': 'calc.py'}),
               ('final', '已读取 calc.py，看到 add 用了减法')])
final, steps, obs = agent_loop(llm, work, tools)
print('最终答复:', final)
print('用了步数:', steps)
print('工具观察(read_file 的真实结果):')
print(obs[0][2])
assert steps == 1 and 'return a - b' in obs[0][2]
print('✅ 端到端跑通：LLM 决定调工具 → 真的读了文件 → 回填 → LLM 给最终答复')

## 5 · 无 key 回退：同一接口，真假 LLM 无缝切换

想接真实模型时，我们提供一个 `get_llm()` 工厂：有 `ANTHROPIC_API_KEY` 且装了 `anthropic` 就用真 Claude（**Messages API**），否则回退 MockLLM。**关键：两者暴露同一个 `step()` 接口**，agent 循环一个字都不用改。

下面的 `ClaudeLLM` 是真实适配的骨架（**无 key 时不会被实例化**，仅作展示；真正调用见模块 05）。

In [ ]:
class ClaudeLLM:
    '''真实 Claude 适配骨架。用 Messages API 的 tool_use/tool_result 协议。
       这里只放接口形态；完整的工具往返在模块 05。'''
    def __init__(self, client, model='claude-sonnet-4-6', tools_schema=None, system=''):
        self.client = client; self.model = model
        self.tools_schema = tools_schema or []; self.system = system
    def step(self, messages):
        resp = self.client.messages.create(
            model=self.model, max_tokens=1024, system=self.system,
            tools=self.tools_schema, messages=messages)
        # resp.stop_reason == 'tool_use' → 返回 ('tool', name, args)；否则 ('final', text)
        for block in resp.content:
            if getattr(block, 'type', None) == 'tool_use':
                return ('tool', block.name, block.input)
        text = ''.join(getattr(b, 'text', '') for b in resp.content)
        return ('final', text)

def get_llm(script=None, model='claude-sonnet-4-6'):
    '''有 key+包 → 真 Claude；否则 → MockLLM。返回对象都有 .step()。'''
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic
            return ClaudeLLM(anthropic.Anthropic(), model=model)
        except ImportError:
            pass
    return MockLLM(script or [('final', '(MockLLM 默认答复)')])

llm = get_llm(script=[('final', 'hello from fallback')])
print('拿到的 LLM 类型:', type(llm).__name__)   # 无 key 时为 MockLLM
assert hasattr(llm, 'step'), '真假 LLM 必须暴露同一个 step() 接口'
print('✅ 无 key 回退就绪：agent 循环对接口编程，真假 LLM 可无缝替换')

In [ ]:
# 清理工作区（用完即焚，不留痕迹）
shutil.rmtree(work, ignore_errors=True)
print('工作区已清理 ✅')

✅ 四件事全部通过，环境与方法论到位。

**本课的契约**：① 用 **MockLLM** 把「LLM↔工具」循环端到端跑通，固定模型决策、专注验证脚手架；② 文件 / shell / pytest 全在 **`tempfile` 临时目录**里**真实执行**，真实但隔离；③ 每个 notebook 都附 **无 key 回退**的真实 Claude 适配，绝不阻断。

**接下来五个模块**：01 文件工具（手）→ 02 Shell 工具（脚）→ 03 代码搜索（眼）→ 04 Edit-Test-Fix 循环（小脑）→ 05 完整编码 Agent（集大成）。每一步都建立在前一步之上。

下一站：**模块 01 · 文件工具**。